In [4]:
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import os

RAW = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\raw"
PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"

# Load reference grid from population raster
with rasterio.open(f"{RAW}/phl_general_2020_geotiff/phl_general_2020.tif") as ref:
    REF_TRANSFORM = ref.transform
    REF_SHAPE     = (ref.height, ref.width)
    REF_CRS       = ref.crs
    REF_PROFILE   = ref.profile.copy()

REF_PROFILE.update({"dtype": "float32", "count": 1, "nodata": -9999})
print(f"Reference grid: {REF_SHAPE} | CRS: {REF_CRS}")

def proximity_raster(gdf, label="layer"):
    """Generate distance raster from vector features. Returns array in degrees."""
    from rasterio.features import rasterize
    import numpy as np
    from scipy.ndimage import distance_transform_edt

    gdf = gdf.to_crs(REF_CRS)

    # Rasterize features as binary (1 = feature present)
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None]
    binary = rasterize(
        shapes,
        out_shape=REF_SHAPE,
        transform=REF_TRANSFORM,
        fill=0,
        dtype=np.uint8
    )

    # Distance transform — gives distance in pixels
    dist_px = distance_transform_edt(1 - binary)

    # Convert pixels to kilometers
    # At Philippine latitudes, 1 degree ≈ 111km
    # Pixel size in degrees
    px_deg = abs(REF_TRANSFORM.a)
    dist_km = dist_px * px_deg * 111.0

    pct_zero = (binary == 1).mean() * 100
    print(f"  ✅ {label}: max={dist_km.max():.1f}km | mean={dist_km.mean():.1f}km")
    return dist_km.astype(np.float32)

def save_proximity(arr, filename):
    path = f"{PROCESSED}/{filename}"
    profile = REF_PROFILE.copy()
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr, 1)
    print(f"  💾 Saved: {filename}")


Reference grid: (61200, 39600) | CRS: EPSG:4326


In [ ]:
# ── 1. Distance to roads ───────────────────────────────────
print("\n[1/5] Distance to roads...")
roads = gpd.read_file(
    f"{RAW}/hotosm_phl_roads_lines_shp/hotosm_phl_roads_lines_shp.shp"
)
print(f"  Unique highway values: {roads['highway'].unique()[:15]}")

major = roads[roads["highway"].isin([
    "motorway", "primary", "secondary", "tertiary",
    "motorway_link", "primary_link", "secondary_link", "trunk", "trunk_link"
])]
print(f"  Major roads: {len(major)} of {len(roads)} total")
dist_road = proximity_raster(major, "Distance to roads")
save_proximity(dist_road, "dist_road.tif")
del roads, major, dist_road

# ── 2. Distance to ports ───────────────────────────────────
print("\n[2/5] Distance to ports...")
ports = gpd.read_file(
    f"{RAW}/hotosm_phl_sea_ports_points_shp/hotosm_phl_sea_ports_points_shp.shp"
)
print(f"  Ports: {len(ports)} features")
dist_port = proximity_raster(ports, "Distance to ports")
save_proximity(dist_port, "dist_port.tif")
del ports, dist_port


Reference grid: (61200, 39600) | CRS: EPSG:4326

[1/5] Distance to roads...
  Unique highway values: <StringArray>
[          'path',        'service',   'unclassified',    'residential',
          'track',       'tertiary',        'footway',      'secondary',
   'construction',     'trunk_link',        'primary',          'steps',
 'secondary_link',          'trunk',     'pedestrian']
Length: 15, dtype: str
  Major roads: 115893 of 1513492 total
  ✅ Distance to roads: max=578.5km | mean=131.1km
  💾 Saved: dist_road.tif

[2/5] Distance to ports...
  Ports: 1112 features
  ✅ Distance to ports: max=599.9km | mean=144.7km
  💾 Saved: dist_port.tif

[3/5] Distance to rivers...
  Rivers: 29888 features


In [2]:
# ── 3. Distance to rivers ──────────────────────────────────
print("\n[3/5] Distance to rivers...")
rivers = gpd.read_file(f"{PROCESSED}/rivers_4326.gpkg")
print(f"  Rivers: {len(rivers)} features")
dist_river = proximity_raster(rivers, "Distance to rivers")
save_proximity(dist_river, "dist_river.tif")
del rivers, dist_river


[3/5] Distance to rivers...
  Rivers: 29888 features
  ✅ Distance to rivers: max=578.9km | mean=136.1km
  💾 Saved: dist_river.tif


In [3]:
# ── 4. Distance to population centers ─────────────────────
print("\n[4/5] Distance to population centers...")
# Extract populated places from admin boundaries (municipality centroids)
adm3 = gpd.read_file(
    f"{RAW}/phl_adm_psa_namria_20231106_shp/phl_admbnda_adm3_psa_namria_20231106.shp"
)
# Use centroids as population center proxies
adm3_centers = adm3.copy()
adm3_centers["geometry"] = adm3.geometry.centroid
print(f"  Municipality centers: {len(adm3_centers)}")
dist_pop = proximity_raster(adm3_centers, "Distance to population centers")
save_proximity(dist_pop, "dist_pop.tif")
del adm3, adm3_centers, dist_pop


[4/5] Distance to population centers...


C:\Users\ailene.nunez\AppData\Local\Temp\ipykernel_24944\377946647.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  adm3_centers["geometry"] = adm3.geometry.centroid


  Municipality centers: 1642
  ✅ Distance to population centers: max=587.0km | mean=134.3km
  💾 Saved: dist_pop.tif


In [4]:
# ── 5. Copy population density ────────────────────────────
print("\n[5/5] Population density...")
import shutil
shutil.copy(
    f"{RAW}/phl_general_2020_geotiff/phl_general_2020.tif",
    f"{PROCESSED}/population_density.tif"
)
print("  ✅ Population density copied to processed/")

print("\n✅ All proximity rasters complete")
print("Files in processed/:")
for f in sorted(os.listdir(PROCESSED)):
    size = os.path.getsize(f"{PROCESSED}/{f}") / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")


[5/5] Population density...
  ✅ Population density copied to processed/

✅ All proximity rasters complete
Files in processed/:
  dem_4326.tif: 4.5 MB
  dist_pop.tif: 9245.7 MB
  dist_port.tif: 9245.7 MB
  dist_river.tif: 9245.7 MB
  dist_road.tif: 9245.7 MB
  excl_coastal.tif: 2311.6 MB
  excl_flood.tif: 2311.6 MB
  excl_landslide.tif: 2311.6 MB
  excl_protected.tif: 2311.6 MB
  excl_seismic.tif: 2311.6 MB
  excl_slope.tif: 2311.6 MB
  excl_volcanic.tif: 2311.6 MB
  faults_phl_4326.gpkg: 0.2 MB
  landslide_cache.pkl: 12303.7 MB
  master_exclusion_mask.tif: 2311.6 MB
  nighttime_lights_phl.tif: 44.8 MB
  population_density.tif: 18490.9 MB
  rivers_4326.gpkg: 26.8 MB


In [6]:
import os
os.remove(f"{PROCESSED}/landslide_cache.pkl")
print("✅ Cache deleted")

✅ Cache deleted


In [ ]:
import rasterio
from rasterio.shutil import copy as rio_copy
import os

PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"

to_compress = [
    "dist_road.tif", "dist_port.tif",
    "dist_river.tif", "dist_pop.tif",
    "excl_coastal.tif", "excl_flood.tif",
    "excl_landslide.tif", "excl_protected.tif",
    "excl_seismic.tif", "excl_slope.tif",
    "excl_volcanic.tif", "master_exclusion_mask.tif",
    "population_density.tif"
]

for fname in to_compress:
    path = f"{PROCESSED}/{fname}"
    tmp  = f"{PROCESSED}/tmp_{fname}"
    try:
        rio_copy(
            path, tmp,
            driver="GTiff",
            compress="lzw",
            tiled=True,
            blockxsize=256,
            blockysize=256,
            bigtiff="YES"
        )
        os.replace(tmp, path)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅ {fname}: {size:.1f} MB")
    except Exception as e:
        print(f"  ⚠ {fname}: {e}")

print("\n✅ Done")

  ✅ dist_road.tif: 8580.9 MB
  ✅ dist_port.tif: 8720.2 MB
  ✅ dist_river.tif: 8366.0 MB
  ✅ dist_pop.tif: 8800.6 MB
  ✅ excl_coastal.tif: 17.3 MB
  ✅ excl_flood.tif: 28.9 MB
  ✅ excl_landslide.tif: 34.3 MB
  ✅ excl_protected.tif: 17.5 MB
  ✅ excl_seismic.tif: 16.9 MB
  ✅ excl_slope.tif: 17.6 MB
  ✅ excl_volcanic.tif: 15.5 MB
  ✅ master_exclusion_mask.tif: 29.9 MB
  ✅ population_density.tif: 165.5 MB

✅ Done


In [6]:
import rasterio
from rasterio.enums import Resampling
from rasterio.shutil import copy as rio_copy
import os

PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"

proximity_files = [
    "dist_road.tif", "dist_port.tif",
    "dist_river.tif", "dist_pop.tif"
]

# Resample factor — 10x downscale (300m → ~1km equivalent)
SCALE = 0.1

for fname in proximity_files:
    path = f"{PROCESSED}/{fname}"
    tmp  = f"{PROCESSED}/tmp_{fname}"
    try:
        with rasterio.open(path) as src:
            # New dimensions
            new_h = int(src.height * SCALE)
            new_w = int(src.width  * SCALE)

            profile = src.profile.copy()
            profile.update({
                "height":    new_h,
                "width":     new_w,
                "transform": src.transform * src.transform.scale(
                    src.width  / new_w,
                    src.height / new_h
                ),
                "compress":  "lzw",
                "tiled":     True,
                "blockxsize": 256,
                "blockysize": 256,
                "bigtiff":   "IF_SAFER"
            })

            data = src.read(
                out_shape=(src.count, new_h, new_w),
                resampling=Resampling.average
            )

        with rasterio.open(tmp, "w", **profile) as dst:
            dst.write(data)
        del data

        os.replace(tmp, path)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅ {fname}: {size:.1f} MB")

    except Exception as e:
        print(f"  ⚠ {fname}: {e}")

print("\n✅ Resampling complete")

  ✅ dist_road.tif: 100.1 MB
  ✅ dist_port.tif: 1.1 MB
  ✅ dist_river.tif: 1.1 MB
  ✅ dist_pop.tif: 1.1 MB

✅ Resampling complete
